<a href="https://colab.research.google.com/github/MrFire24/Dataset-Organizer/blob/main/dataset_organizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from logging import warn
from google.colab import files
import pandas as pd
from pathlib import Path

# --- Настройки ---

OUTPUT_PATH = Path('processed_data')
STRIP_PREFIXES = True
IMPORT_FROM_PATH = True
IMPORT_PATH = Path('uploaded_data/dataset_all_fields.csv')

# --- Функции ---

def strip_prefixes(df, group):
    return df.rename(columns=lambda col: col.removeprefix(group + '_') if col != SYS_TIME else col)

SYS_TIME = 'sys_time'
GROUPS = ['coordinate', 'io', 'control', 'weld', 'scanner', 'termo', 'set', 'positioner']

def process_csv(file):
  df = pd.read_csv(file)
  for group in GROUPS:
      cols = [SYS_TIME] + [col for col in df.columns if col.startswith(group + '_')]
      if len(cols) > 1:
          result = df[cols]
          if STRIP_PREFIXES:
              result = strip_prefixes(result, group)
          result.to_csv(OUTPUT_PATH / f'{group}.csv', index=False)

def process_json(file):
  pass

# --- Мэйн ---
if (IMPORT_FROM_PATH):
  process_csv(IMPORT_PATH)
else:
  OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
  uploaded = files.upload('uploaded_data')

  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
  print()

  for file in uploaded:
    if file.lower().endswith('.csv'):
      process_csv(file)
    elif file.lower().endswith('.json'):
      process_json(file)
    else:
      warn(f'Unsupported file type: {file}')


FileNotFoundError: [Errno 2] No such file or directory: 'uploaded_data/dataset_all_fields.csv'

In [7]:

# @title Получение файла {"single-column":true, run:"auto"}

source_type = "Google Drive Link" #@param ["Upload File", "Google Drive", "Google Drive Link"]
#add_job_json == False #@param {type:"boolean"}
# @markdown Выбирав опцию "Google Drive" будет запрошен доспут к вашему диску

# @markdown Также в этом случае файл должен находится на вашем Google диске

from IPython.core.interactiveshell import dis
import ipywidgets as widgets
from IPython.display import display
import os

# Файл который мы пытаемся получить для дальнейшего парсинга
data_file_path = ""

##########################

def get_file_local():
  from google.colab import files
  print("\nВнимание!")
  print("Загрузка больших файлов таким методом может занять много времени")
  print("Повторная загрузка файла создаёт дубликат\n")

  upl_files = files.upload('uploaded_data')
  if len(upl_files.keys()) > 1:
    print("Слишком много файлов!")
  else:
    print(upl_files.keys())
    full_path = "/content/" + list(upl_files.keys())[0]
    if os.path.exists(full_path) and full_path.lower().endswith('.csv'):
      global data_file_path
      data_file_path = full_path
      print("Файл успешно загружен!")
    else:
      print("Файл не найден! Что-то пошло не так")
      print(full_path)

##########################

def extract_file_id(value):
    if '/d/' in value:
        return value.split('/d/')[1].split('/')[0]
    return value

file_link_input = widgets.Text(placeholder="Введите ссылку")
confirm_button_link = widgets.Button(description="Подтвердить", icon='check')
def on_confirm_button_link(button):
  import gdown
  file_link = extract_file_id(file_link_input.value)
  url = f'https://drive.google.com/uc?id={file_link}'
  output = "uploaded_data/"

  full_path = ""
  try:
    full_path = "/content/" + gdown.download(url, output, quiet=False)
  except:
    print("Файл недоступен или указано неверное ссылка")
    button.button_style = "danger"
  else:
    print()
    if os.path.exists(full_path):
      if full_path.lower().endswith('.csv'):
        print("Файл успешно загружен!")
        button.button_style = "success"
        global data_file_path
        data_file_path = full_path
        print("Вы можете запустить следующий блок кода для начала парсинга")
      else:
        print("Файл найден, но это НЕ CSV файл")
        button.button_style = "warning"
    else:
      print("Файл не найден! Что-то пошло не так")
      print(full_path)
      button.button_style = "danger"

def get_file_from_drive_link():
  # Создание виджета ввода
  print("Введите ссылку на файл с Google Drive")
  print("Из ссылки будет получено ID файла для скачивания")
  print("Убедитесь что в ссылке есть \"d/\"")
  confirm_button_link.on_click(on_confirm_button_link)
  display(file_link_input, confirm_button_link)

########################

file_path_input = widgets.Text(placeholder="Введите путь к файлу")
confirm_button_path = widgets.Button(description="Подтвердить", icon='check')
def on_confirm_button_path(button):
  full_path = "/content/drive/MyDrive/" + file_path_input.value
  if os.path.exists(full_path):
    if full_path.lower().endswith('.csv'):
      print("Файл найден")
      button.button_style = "success"
      global data_file_path
      data_file_path = full_path
      print("Вы можете запустить следующий блок кода для начала парсинга")
    else:
      print("Файл найден, но это НЕ CSV файл")
      button.button_style = "warning"
  else:
    print("Файл не найден по указанному пути")
    print("/content/drive/" + file_path_input.value)
    button.button_style = "danger"

def get_file_from_drive():
  # Подключение к диску
  from google.colab import drive
  drive.mount("/content/drive", force_remount=True)

  # Создание виджета ввода
  print("Введите путь к вашему файлу (filename.csv)")
  print("Если он находиться в какой-то папке на диске то путь должен вышлядить так:")
  print("\"folder/filename.csv\"")

  confirm_button_path.on_click(on_confirm_button_path)
  display(file_path_input, confirm_button_path)


################

if source_type == "Upload File":
  get_file_local()
elif source_type == "Google Drive":
  get_file_from_drive()
elif source_type == "Google Drive Link":
  get_file_from_drive_link()


Введите ссылку на файл с Google Drive
Из ссылки будет получено ID файла для скачивания
Убедитесь что в ссылке есть "d/"


Text(value='', placeholder='Введите ссылку')

Button(description='Подтвердить', icon='check', style=ButtonStyle())

Downloading...
From (original): https://drive.google.com/uc?id=1QI3Zo4hauR0z_8TZSZsj1YYkLesmI3uY
From (redirected): https://drive.google.com/uc?id=1QI3Zo4hauR0z_8TZSZsj1YYkLesmI3uY&confirm=t&uuid=98463b27-5d7f-4cf7-a7be-c5719265b9e9
To: /content/uploaded_data/dataset_all_fields.csv
100%|██████████| 208M/208M [00:02<00:00, 84.8MB/s]


Файл успешно загружен!
Вы можете запустить следующий блок кода для начала парсинга


In [10]:
# @title Парсинг {"single-column":true}

#jbi_name
CUSTOM_DATA_PATH = "" #@param {type:"string", placeholder:"Оставьте пустым если использовали блок выше"}
CUSTOM_NAME = "" #@param {type:"string", placeholder:"По стандарту берёт первое значение jbi_name"}
# @markdown ---
OUTPUT_PATH = "processed_data" #@param {type:"string"}
STRIP_PREFIXES = True #@param {type:"boolean"}

from google.colab import files
import pandas as pd
from pathlib import Path

# Гарантируем наличие папки для вывода
os.makedirs(OUTPUT_PATH, exist_ok=True)

def strip_prefixes(df, group):
    return df.rename(columns=lambda col: col.removeprefix(group + '_') if col != SYS_TIME else col)

def get_jbi_name(df):
    if 'control_jbi_name' in df.columns:
        name = df['control_jbi_name'].iloc[0]
        if not pd.isna(name) and str(name).strip() != '':
            return str(name).strip()
    return 'UNNAMED'

SYS_TIME = 'sys_time'
GROUPS = ['coordinate', 'io', 'control', 'weld', 'scanner', 'termo', 'set', 'positioner']

def process_csv(file):
  df = pd.read_csv(file)

  jbi_name = CUSTOM_NAME.strip() if CUSTOM_NAME.strip() else get_jbi_name(df)
  start_time = pd.to_datetime(df[SYS_TIME].iloc[0]).strftime('%Y-%m-%d_%H-%M-%S')

  session_path = Path(OUTPUT_PATH) / jbi_name / start_time

  if session_path.exists():
    answer = input(f"Папка {session_path} уже существует. Перезаписать? (y/n): ")
    if answer.lower() != 'y':
        print("Парсинг отменён.")
        return False

  session_path.mkdir(parents=True, exist_ok=True)

  for group in GROUPS:
      cols = [SYS_TIME] + [col for col in df.columns if col.startswith(group + '_')]
      if len(cols) > 1:
          result = df[cols]
          if STRIP_PREFIXES:
              result = strip_prefixes(result, group)
          result.to_csv(session_path / f'{group}.csv', index=False)
  return True

def process_json(file):
  pass

# --- Мэйн ---
if 'data_file_path' not in dir() or not data_file_path:
    data_file_path = None

if CUSTOM_DATA_PATH:
    data_file_path = "/content/" + CUSTOM_DATA_PATH

if data_file_path and Path(data_file_path).exists():
    if process_csv(data_file_path):
        print("Парсинг завершен!")
else:
    print("Ошибка: Путь к файлу пуст или файл не найден.")
    if not CUSTOM_DATA_PATH:
        print("Сначала загрузите файл в блоке выше.")
    else:
        print("Проверьте указанный вами путь.")
    print(data_file_path)

Парсинг завершен!


In [ ]:
import pandas as pd
df = pd.read_csv(data_file_path)
display(df)

,sys_time,set_frame,set_tool,set_dz,set_alc,set_pdc,set_wfs,set_speed,set_job_number,set_temp_treshold,...,scanner_timestamp,scanner_duration,scanner_xz_max,scanner_point_count,scanner_x,scanner_z,scanner_i,termo_timestamp,termo_duration,termo_temperature
0,2026-01-08T08:21:14.861573Z,6,8,0.6,4.8,4.9,3.0,110,33,150.0,...,2026-01-08T08:21:14.861573Z,1.2,"[2.2, 79.0]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[94.88, 106.84, 145.69, 153.78, 148.9, 79.51, ...","[143, 407, 567, 780, 380, 876, 646, 684, 198, ...",2026-01-08T08:21:14.861573Z,129.3,181.0
1,2026-01-08T08:21:14.881573Z,6,8,0.6,4.8,4.9,3.0,110,33,150.0,...,2026-01-08T08:21:14.881573Z,1.2,"[2.2, 80.9]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[113.91, 148.06, 81.27, 126.62, 135.43, 92.9, ...","[455, 571, 537, 402, 843, 800, 778, 338, 140, ...",2026-01-08T08:21:14.881573Z,129.3,179.5
2,2026-01-08T08:21:14.901573Z,6,8,0.6,4.8,4.9,3.0,110,33,150.0,...,2026-01-08T08:21:14.901573Z,1.2,"[2.8, 80.4]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[80.33, 102.89, 128.35, 100.9, 110.75, 90.59, ...","[703, 384, 102, 421, 942, 886, 952, 532, 890, ...",2026-01-08T08:21:14.901573Z,129.3,183.7
3,2026-01-08T08:21:14.921573Z,6,8,0.6,4.8,4.9,3.0,110,33,150.0,...,2026-01-08T08:21:14.921573Z,1.2,"[2.9, 80.9]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[130.84, 143.42, 148.57, 100.07, 85.28, 144.04...","[409, 386, 825, 330, 135, 483, 301, 763, 938, ...",2026-01-08T08:21:14.921573Z,129.3,176.0
4,2026-01-08T08:21:14.941573Z,6,8,0.6,4.8,4.9,3.0,110,33,150.0,...,2026-01-08T08:21:14.941573Z,1.2,"[2.9, 80.5]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[110.56, 76.84, 151.33, 91.03, 139.51, 151.45,...","[620, 300, 562, 654, 456, 466, 366, 735, 88, 1...",2026-01-08T08:21:14.941573Z,129.3,184.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2026-01-08T08:24:34.761573Z,5,8,0.6,4.6,4.4,3.0,112,33,250.0,...,2026-01-08T08:24:34.761573Z,1.0,"[2.8, 80.5]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[88.17, 150.21, 120.61, 99.67, 126.65, 76.8, 1...","[492, 200, 703, 165, 211, 756, 299, 585, 39, 2...",2026-01-08T08:24:34.761573Z,121.2,179.4
9996,2026-01-08T08:24:34.781573Z,5,8,0.6,4.6,4.4,3.0,112,33,250.0,...,2026-01-08T08:24:34.781573Z,1.0,"[2.4, 80.6]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[151.82, 142.64, 103.79, 101.71, 122.96, 107.5...","[966, 970, 952, 998, 676, 308, 485, 223, 668, ...",2026-01-08T08:24:34.781573Z,121.2,183.2
9997,2026-01-08T08:24:34.801573Z,5,8,0.6,4.6,4.4,3.0,112,33,250.0,...,2026-01-08T08:24:34.801573Z,1.0,"[2.4, 80.0]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[136.73, 121.56, 84.43, 124.05, 87.84, 136.38,...","[43, 398, 780, 2, 573, 118, 615, 677, 35, 159,...",2026-01-08T08:24:34.801573Z,121.2,178.9
9998,2026-01-08T08:24:34.821573Z,5,8,0.6,4.6,4.4,3.0,112,33,250.0,...,2026-01-08T08:24:34.821573Z,1.0,"[2.7, 79.6]",1028,"[-50.0, -49.9, -49.81, -49.71, -49.61, -49.51,...","[123.18, 140.93, 114.63, 74.32, 129.36, 143.68...","[43, 986, 407, 489, 67, 140, 540, 145, 260, 45...",2026-01-08T08:24:34.821573Z,121.2,177.4
